In [2]:
import osmnx as ox
import geopandas as gpd
import networkx as nx
import pandas as pd
from shapely.geometry import Point, MultiPoint
from shapely.ops import unary_union
import os, time
from pyrosm import OSM
from shapely.geometry import Polygon

BASE = "/home/jovyan/work/children15mc"
RAW = f"{BASE}/data/raw"
PROC = f"{BASE}/data/processed"
ISO_DIR = f"{PROC}/iso_barrier_by_borough" 
LOCAL_PBF = f"{RAW}/greater-london-260812.osm.pbf"
os.makedirs(ISO_DIR, exist_ok=True)

ox.settings.use_cache = True
ox.settings.cache_folder = f"{BASE}/cache"

CRS = 27700
WALK_SPEED_KMH = 3.5
TRIP_MIN = 15
m_per_min = WALK_SPEED_KMH * 1000 / 60

MAJOR = "motorway|trunk|primary|secondary|tertiary"
VALID_CROSSING = ["traffic_signals", "marked", "uncontrolled"]
CROSS_TOL = 15 

def make_isochrone(G, x, y, trip_min=TRIP_MIN, buff=30):
    pt = gpd.GeoSeries([Point(x, y)], crs=CRS).to_crs(4326).iloc[0]
    try:
        node = ox.distance.nearest_nodes(G, pt.x, pt.y)
    except Exception:
        return None
    sub = nx.ego_graph(G, node, radius=trip_min, distance="walk_time")
    pts = [Point(d["x"], d["y"]) for _, d in sub.nodes(data=True)]
    if len(pts) < 3:
        return None
    poly = gpd.GeoSeries(MultiPoint(pts), crs=4326).to_crs(CRS)
    return poly.convex_hull.buffer(buff).iloc[0]

print("Configuration complete.")

Configuration complete.


In [3]:
def build_barrier_network(G, poly_4326,osm):
    
    edges = ox.graph_to_gdfs(G, nodes=False).to_crs(CRS)
    hw = edges["highway"].astype(str)
    remove = set()

    # Filter 1: The major road itself
    is_major = hw.str.contains(MAJOR, na=False)
    remove |= set(edges[is_major].index)

    # Filter 2: Unprotected crossings
    major_geom = unary_union(edges[is_major].geometry) if is_major.any() else None
    if major_geom is not None:
        
        try:
            cross_data = osm.get_data_by_custom_criteria(
                custom_filter={"highway": ["crossing"]},
                extra_attributes=["crossing"]
            )
            
            if cross_data is not None:
                cross = cross_data[cross_data.geometry.type == "Point"].to_crs(CRS)
            else:
                cross = gpd.GeoDataFrame(geometry=[], crs=CRS)
                
            if "crossing" in cross.columns:
                safe = cross[cross["crossing"].astype(str).isin(VALID_CROSSING)]
            else:
                safe = cross.iloc[0:0]
        except Exception:
            safe = gpd.GeoDataFrame(geometry=[], crs=CRS)

        safe_buf = unary_union(safe.geometry.buffer(CROSS_TOL)) if len(safe) else None

        non_major = edges[~is_major]
        touches = non_major.geometry.intersects(major_geom)
        for idx, g in non_major[touches].geometry.items():
            ip = g.intersection(major_geom)
            if ip.is_empty:
                continue
            if safe_buf is None or not ip.intersects(safe_buf):
                remove.add(idx)

    # Filter 3: Railway barriers without bridges or tunnels
    try:
        rail_data = osm.get_data_by_custom_criteria(custom_filter={"railway": ["rail"]})
        if rail_data is not None:
            rail = rail_data[rail_data.geometry.type.isin(["LineString", "MultiLineString"])].to_crs(CRS)
            rail_geom = unary_union(rail.geometry) if len(rail) else None
        else:
            rail_geom = None
    except Exception:
        rail_geom = None

    if rail_geom is not None:
        bridge = edges["bridge"].astype(str) if "bridge" in edges.columns else pd.Series("nan", index=edges.index)
        tunnel = edges["tunnel"].astype(str) if "tunnel" in edges.columns else pd.Series("nan", index=edges.index)
        protected = bridge.str.contains("yes|viaduct", na=False) | tunnel.str.contains("yes", na=False)
        crosses_rail = edges.geometry.intersects(rail_geom)
        remove |= set(edges[crosses_rail & ~protected].index)

    keep = [e for e in edges.index if e not in remove]
    G_b = G.edge_subgraph(keep).copy()
    return G_b, len(remove), len(edges)

print("Function ready")

Function ready


In [7]:
df = gpd.read_file(f"{PROC}/analysis_table.gpkg")
boroughs = sorted(df["lad22nm"].unique())
print(f"Total: {len(boroughs)} boroughs, {len(df)} LSOAs\n")

log = []
for bname in boroughs:
    safe_name = bname.replace(" ", "_").replace(",", "")
    outfile = f"{ISO_DIR}/{safe_name}.gpkg"

    if os.path.exists(outfile):
        print(f"✓ {bname}, Skipped")
        continue

    t0 = time.time()
    sub = df[df["lad22nm"] == bname].copy()
    t0 = time.time()
    sub = df[df["lad22nm"] == bname].copy()

    # Network
    poly_27700 = sub.geometry.union_all().buffer(875)
    poly_4326 = gpd.GeoSeries([poly_27700], crs=CRS).to_crs(4326).iloc[0]
    try:
        bbox = list(poly_4326.bounds) 
        
        osm = OSM(LOCAL_PBF, bounding_box=bbox)
        nodes, edges = osm.get_network(network_type="walking", nodes=True)
        
        G = osm.to_graph(nodes, edges, graph_type="networkx")
        
    except Exception as e:
        print(f"✗ {bname} Failed: {e}")
        continue

    G_b, n_removed, n_total = build_barrier_network(G, poly_4326,osm)

    for u, v, k, d in G_b.edges(keys=True, data=True):
        d["walk_time"] = d["length"] / m_per_min

    rows = []
    for _, r in sub.iterrows():
        geom = make_isochrone(G_b, r.cent_x, r.cent_y)
        rows.append({"lsoa21cd": r.lsoa21cd, "geometry": geom})

    iso = gpd.GeoDataFrame(rows, crs=CRS).dropna(subset=["geometry"])
    iso.to_file(outfile, driver="GPKG")

    pct = 100 * n_removed / n_total
    log.append({"borough": bname, "removed": n_removed, "total": n_total, "pct": pct})
    print(f"✓ {bname}: Disconnected {n_removed}/{n_total} ({pct:.1f}%), Isochrones {len(iso)}/{len(sub)}, {time.time()-t0:.0f}s")

pd.DataFrame(log).to_csv(f"{PROC}/barrier_removal_log.csv", index=False)
print("All done!")

Total: 33 boroughs, 4994 LSOAs

✓ Barking and Dagenham, Skipped
✓ Barnet, Skipped
✓ Bexley, Skipped
✓ Brent, Skipped
✓ Bromley, Skipped
✓ Camden, Skipped
✓ City of London, Skipped
✓ Croydon, Skipped
✓ Ealing, Skipped
✓ Enfield, Skipped
✓ Greenwich, Skipped
✓ Hackney, Skipped
✓ Hammersmith and Fulham, Skipped
✓ Haringey, Skipped
✓ Harrow, Skipped
✓ Havering, Skipped
✓ Hillingdon, Skipped
✓ Hounslow, Skipped
✓ Islington, Skipped
✓ Kensington and Chelsea, Skipped
✓ Kingston upon Thames, Skipped
✓ Lambeth, Skipped
✓ Lewisham, Skipped
✓ Merton, Skipped
✓ Newham, Skipped
✓ Redbridge, Skipped
✓ Richmond upon Thames, Skipped
✓ Southwark, Skipped
✓ Sutton, Skipped
✓ Tower Hamlets, Skipped
✓ Waltham Forest, Skipped
✓ Wandsworth, Skipped
✓ Westminster, Skipped
All done!


In [8]:
log = []

for bname in boroughs:
    t0 = time.time()
    sub = df[df["lad22nm"] == bname].copy()

    poly_27700 = sub.geometry.union_all().buffer(875)
    poly_4326 = gpd.GeoSeries([poly_27700], crs=CRS).to_crs(4326).iloc[0]

    try:
        bbox = list(poly_4326.bounds)
        osm = OSM(LOCAL_PBF, bounding_box=bbox)
        nodes, edges = osm.get_network(network_type="walking", nodes=True)
        G = osm.to_graph(nodes, edges, graph_type="networkx")
    except Exception as e:
        print(f"✗ {bname} Failed: {e}")
        continue

    _, n_removed, n_total = build_barrier_network(G, poly_4326, osm)
    pct = 100 * n_removed / n_total

    log.append({"borough": bname, "removed": n_removed, "total": n_total, "pct": pct})
    print(f"✓ {bname}: Disconnected {n_removed}/{n_total} ({pct:.1f}%), {time.time()-t0:.1f}s")

pd.DataFrame(log).to_csv(f"{PROC}/barrier_removal_log.csv", index=False)
print("\n Saved barrier_removal_log.csv")

✓ Barking and Dagenham: Disconnected 21660/158350 (13.7%), 65.1s
✓ Barnet: Disconnected 44082/340662 (12.9%), 200.6s
✓ Bexley: Disconnected 29920/209986 (14.2%), 94.7s
✓ Brent: Disconnected 32600/230572 (14.1%), 136.0s
✓ Bromley: Disconnected 65978/493200 (13.4%), 387.4s
✓ Camden: Disconnected 25598/330698 (7.7%), 148.2s
✓ City of London: Disconnected 6412/158550 (4.0%), 43.5s
✓ Croydon: Disconnected 54200/399522 (13.6%), 277.0s
✓ Ealing: Disconnected 32540/264798 (12.3%), 142.6s
✓ Enfield: Disconnected 31168/247622 (12.6%), 112.9s
✓ Greenwich: Disconnected 29048/421024 (6.9%), 151.8s
✓ Hackney: Disconnected 15494/283498 (5.5%), 96.2s
✓ Hammersmith and Fulham: Disconnected 18344/216178 (8.5%), 91.5s
✓ Haringey: Disconnected 17858/184510 (9.7%), 67.1s
✓ Harrow: Disconnected 26734/175876 (15.2%), 79.2s
✓ Havering: Disconnected 24290/158686 (15.3%), 70.9s
✓ Hillingdon: Disconnected 41344/301586 (13.7%), 164.4s
✓ Hounslow: Disconnected 43636/343454 (12.7%), 199.7s
✓ Islington: Disconnected

In [ ]:
log_file = f"{PROC}/barrier_removal_log.csv"

if os.path.exists(log_file):
    log_df = pd.read_csv(log_file)
    for _, row in log_df.iterrows():
        print(f"✓ {row['borough']}: Disconnected {int(row['removed'])}/{int(row['total'])} ({row['pct']:.1f}%)")
else:
    print("File not found.")

In [4]:
import glob
files = glob.glob(f"{ISO_DIR}/*.gpkg")
print(f"Completed {len(files)}/33")
all_iso = pd.concat([gpd.read_file(f) for f in files], ignore_index=True)
all_iso = gpd.GeoDataFrame(all_iso, crs=CRS)
all_iso.to_file(f"{PROC}/isochrones_barrier.gpkg", driver="GPKG")
print(f"Total isochrones: {len(all_iso)}")

Completed 33/33
Total isochrones: 4964


In [5]:
iso_d = gpd.read_file(f"{PROC}/isochrones_all.gpkg")
iso_b = gpd.read_file(f"{PROC}/isochrones_barrier.gpkg")

print(f"Default network: {len(iso_d)} isochrones, average area {iso_d.area.mean()/1e6:.3f} km²")
print(f"Barrier network: {len(iso_b)} isochrones, average area {iso_b.area.mean()/1e6:.3f} km²")

cmp = iso_d[["lsoa21cd"]].copy()
cmp["area_default"] = iso_d.area / 1e6
cmp = cmp.merge(
    iso_b[["lsoa21cd"]].assign(area_barrier=iso_b.area.values / 1e6),
    on="lsoa21cd", how="inner")

cmp["area_loss"] = cmp["area_default"] - cmp["area_barrier"]
cmp["loss_pct"] = 100 * cmp["area_loss"] / cmp["area_default"]

print(f"\nMatched LSOAs: {len(cmp)}")
print(f"Average area loss: {cmp['area_loss'].mean():.3f} km² ({cmp['loss_pct'].mean():.1f}%)")
print(f"Median loss percentage: {cmp['loss_pct'].median():.1f}%")
print(f"LSOAs with >30% loss: {(cmp['loss_pct']>30).sum()} ({100*(cmp['loss_pct']>30).mean():.1f}%)")

cmp.to_csv(f"{PROC}/area_comparison.csv", index=False)

Default network: 4994 isochrones, average area 1.239 km²
Barrier network: 4964 isochrones, average area 0.664 km²

Matched LSOAs: 4964
Average area loss: 0.575 km² (45.4%)
Median loss percentage: 44.5%
LSOAs with >30% loss: 3213 (64.7%)
